# Polymarket API Explorer

Covers **every major public endpoint** across the three official Polymarket APIs:

1. **Gamma API** (`https://gamma-api.polymarket.com`)
   - Markets, events, tags, series, sports, search, profiles.
   - Fully public, no auth needed.

2. **Data API** (`https://data-api.polymarket.com`)
   - Positions, trades, activity, open interest, holders, leaderboards, analytics.
   - Fully public.

3. **CLOB API** (`https://clob.polymarket.com`)
   - Order books, prices, midpoints, spreads, last trades, tick sizes, price history.
   - Public read endpoints (trading requires auth — noted below).

**Note on real-time data:** For live orderbook/price streams, prefer the
WebSocket API (`wss://ws-subscriptions-clob.polymarket.com/ws/market`) over
polling these REST endpoints.

In [1]:
import json
import time
import logging
from typing import Any, Optional

import requests
import pandas as pd
from pprint import pprint
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("polymarket")

# ── Display options ────────────────────────────────────────────────────────────
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


In [3]:
# =============================================================================
# Rate-limiter  (simple token-bucket — default: 5 req/s, well under the limit)
# =============================================================================
class RateLimiter:
    """Minimal token-bucket rate limiter."""

    def __init__(self, calls_per_second: float = 5.0):
        self.min_interval = 1.0 / calls_per_second
        self._last_call: float = 0.0

    def wait(self) -> None:
        elapsed = time.monotonic() - self._last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_call = time.monotonic()


In [4]:
# =============================================================================
# Unified HTTP client  (retry + exponential back-off baked in)
# =============================================================================
class PolymarketClient:
    """
    Thin wrapper around requests.Session with:
      - automatic retry + exponential back-off (handles 429 / 5xx)
      - shared rate-limiter
      - consistent error messages
    """

    GAMMA_BASE = "https://gamma-api.polymarket.com"
    DATA_BASE  = "https://data-api.polymarket.com"
    CLOB_BASE  = "https://clob.polymarket.com"

    def __init__(self, calls_per_second: float = 5.0, timeout: int = 20):
        self.timeout = timeout
        self._limiter = RateLimiter(calls_per_second)
        self._session = self._build_session()

    # ── internals ─────────────────────────────────────────────────────────────
    @staticmethod
    def _build_session() -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=4,
            backoff_factor=0.5,          # waits 0.5, 1, 2, 4 s between retries
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"],
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("https://", adapter)
        session.mount("http://",  adapter)
        return session

    def get(
        self,
        url: str,
        params: Optional[dict] = None,
    ) -> Optional[Any]:
        """GET `url`, return parsed JSON or None on failure."""
        self._limiter.wait()
        try:
            r = self._session.get(url, params=params, timeout=self.timeout)
            r.raise_for_status()
            return r.json()
        except requests.HTTPError:
            log.error("HTTP %s  GET %s", r.status_code, url)
        except Exception as exc:
            log.error("Request failed  GET %s  → %s", url, exc)
        return None

    def post(
        self,
        url: str,
        data: Optional[Any] = None,
        json_data: Optional[Any] = None,
    ) -> Optional[Any]:
        """POST `url`, return parsed JSON or None on failure."""
        self._limiter.wait()
        try:
            r = self._session.post(url, data=data, json=json_data, timeout=self.timeout)
            r.raise_for_status()
            return r.json()
        except requests.HTTPError:
            log.error("HTTP %s  POST %s", r.status_code, url)
        except Exception as exc:
            log.error("Request failed  POST %s  → %s", url, exc)
        return None

    # ── convenience builders ──────────────────────────────────────────────────
    def gamma(self, path: str, **params) -> Optional[Any]:
        return self.get(f"{self.GAMMA_BASE}{path}", params=params or None)

    def data(self, path: str, **params) -> Optional[Any]:
        return self.get(f"{self.DATA_BASE}{path}", params=params or None)

    def clob(self, path: str, **params) -> Optional[Any]:
        return self.get(f"{self.CLOB_BASE}{path}", params=params or None)


client = PolymarketClient()

In [5]:
# =============================================================================
# Helpers
# =============================================================================
def parse_token_ids(raw) -> list[str]:
    """Return a flat list of CLOB token-IDs from clobTokenIds (str or list)."""
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return [t.strip() for t in raw.split(",") if t.strip()]
    if isinstance(raw, list):
        return raw
    return []


def safe_df(data, cols: list[str]) -> pd.DataFrame:
    """Build a DataFrame and silently drop columns that are absent."""
    df = pd.DataFrame(data) if data else pd.DataFrame()
    keep = [c for c in cols if c in df.columns]
    return df[keep] if keep else df


def parse_outcome_prices(raw) -> list[float]:
    """outcomePrices is returned as a JSON string — parse it."""
    if isinstance(raw, str):
        try:
            return [float(p) for p in json.loads(raw)]
        except (json.JSONDecodeError, ValueError):
            pass
    if isinstance(raw, list):
        return [float(p) for p in raw]
    return []

## 1.  GAMMA API
Primary source for market discovery and metadata.


### 1.1  Markets — top 10 by 24 h volume

In [6]:
markets_raw = client.gamma(
    "/markets",
    limit=10,
    order="volume24hr",
    ascending="false",
    active="true",
    closed="false",
)

markets_df = safe_df(
    markets_raw,
    ["id", "question", "slug", "active", "closed", "volume24hr", "liquidity",
     "outcomePrices", "clobTokenIds"],
)

if not markets_df.empty:
    # Parse outcomePrices from JSON string → list[float]
    if "outcomePrices" in markets_df.columns:
        markets_df["outcomePrices"] = markets_df["outcomePrices"].apply(parse_outcome_prices)

    print("=== Top 10 Markets by 24 h Volume ===")
    display(markets_df)
    first_market = markets_raw[0]
else:
    print("No markets returned.")
    first_market = {}

=== Top 10 Markets by 24 h Volume ===


,id,question,slug,active,closed,volume24hr,liquidity,outcomePrices,clobTokenIds
0,1640919,US forces enter Iran by April 30?,us-forces-enter-iran-by-april-30-899,True,False,3.591716e+07,3287598.32722,"[0.9975, 0.0025]","[""2916184120206223749839849644877707470354946028257066951797428049170871002238"", ""7653310878196227531065116514963407..."
1,1394299,US forces enter Iran by December 31?,us-forces-enter-iran-by-december-31-573-642-385-371-179-425-262,True,False,3.584064e+06,729364.01725,"[0.9985, 0.0015]","[""31335564527155177318544135513783493075328451393660649396114225549873718295223"", ""999431386086915652803375764895435..."
2,1455604,Will Trump talk to Xi Jinping in March?,will-trump-talk-to-xi-jinping-in-march-165,True,False,3.184791e+06,796545.14371,"[0.0015, 0.9985]","[""97799548617883562371099987524353846715161843398499100552568275384882002822021"", ""485360711488650258779956610756483..."
3,1856503,Spread: Suns (-10.5),nba-phx-chi-2026-04-05-spread-away-10pt5,True,False,2.226435e+06,363843.18867,"[0.0005, 0.9995]","[""79591019971442663809805459411811365131381618283403311206481165395313860824571"", ""688493463761015473143647023767183..."
4,1706788,US x Iran ceasefire by April 7?,us-x-iran-ceasefire-by-april-7-278,True,False,2.091742e+06,224933.26305,"[0.0235, 0.9765]","[""82855088893985825781350466813737280564000275725006328179621744619327480699369"", ""551947454530742975609004389083577..."
5,1791478,Raptors vs. Celtics,nba-tor-bos-2026-04-05,True,False,2.003421e+06,991227.13863,"[0.0005, 0.9995]","[""46314988444372811517520002071803534269341567798463321407996241864916562503066"", ""424683624249688497005787532210178..."
6,669660,Will the Fed decrease interest rates by 50+ bps after the April 2026 meeting?,will-the-fed-decrease-interest-rates-by-50-bps-after-the-april-2026-meeting,True,False,1.825477e+06,2248890.57859,"[0.0035, 0.9965]","[""18690049947242812495755151360212639738977254879109748949267393375856311641700"", ""442177543606339796803169897698995..."
7,1869939,Spread: Bucks (-6.5),nba-mem-mil-2026-04-05-spread-home-6pt5,True,False,1.708527e+06,248767.08876,"[0.9995, 0.0005]","[""75271331225589719661157517835674839204611962333409221602882272302777750238305"", ""508344326272644845968564536800423..."
8,558958,Will Australia win the 2026 FIFA World Cup?,will-australia-win-the-2026-fifa-world-cup-816,True,False,1.601899e+06,1887227.74652,"[0.0025, 0.9975]","[""43661509251351142169227141691164122649250455115438867334436875294380701133091"", ""367263712204357224627344672181029..."
9,1537972,Hurricanes vs. Senators,nhl-car-ott-2026-04-05,True,False,1.548527e+06,12061.1237,"[0.395, 0.605]","[""71692652511729592627525300040642964799150387499986979173208609060668139526585"", ""355730529380013865870986466502316..."


### 1.2  Single Market — by ID and by slug

In [7]:
if first_market:
    market_id   = first_market["id"]
    market_slug = first_market.get("slug")

    # ── by ID ──────────────────────────────────────────────────────────────
    detail_by_id = client.gamma(f"/markets/{market_id}")
    if detail_by_id:
        print(f"\n── Market detail (id={market_id}) ──")
        pprint({
            k: detail_by_id[k]
            for k in ["id", "question", "slug", "outcomes", "clobTokenIds", "endDate"]
            if k in detail_by_id
        })

    # ── by slug ────────────────────────────────────────────────────────────
    if market_slug:
        detail_by_slug = client.gamma("/markets", slug=market_slug)
        if detail_by_slug and isinstance(detail_by_slug, list):
            print(f"\n── Market detail (slug={market_slug}) ──")
            pprint({
                k: detail_by_slug[0][k]
                for k in ["id", "question", "slug", "clobTokenIds"]
                if k in detail_by_slug[0]
            })


── Market detail (id=1640919) ──
{'clobTokenIds': '["2916184120206223749839849644877707470354946028257066951797428049170871002238", '
                 '"76533108781962275310651165149634079251899733930834190485860627580128626747247"]',
 'endDate': '2026-04-30T00:00:00Z',
 'id': '1640919',
 'outcomes': '["Yes", "No"]',
 'question': 'US forces enter Iran by April 30?',
 'slug': 'us-forces-enter-iran-by-april-30-899'}

── Market detail (slug=us-forces-enter-iran-by-april-30-899) ──
{'clobTokenIds': '["2916184120206223749839849644877707470354946028257066951797428049170871002238", '
                 '"76533108781962275310651165149634079251899733930834190485860627580128626747247"]',
 'id': '1640919',
 'question': 'US forces enter Iran by April 30?',
 'slug': 'us-forces-enter-iran-by-april-30-899'}


### 1.3  Events — top 8 by volume

In [8]:
events_raw = client.gamma(
    "/events",
    limit=8,
    active="true",
    closed="false",
    order="volume",
)

print("=== Top Events ===")
display(safe_df(
    events_raw,
    ["id", "title", "slug", "active", "marketsCount", "volume24hr"],
))

=== Top Events ===


,id,title,slug,active,volume24hr
0,348245,"Dogecoin Up or Down - April 6, 6:40PM-6:45PM ET",doge-updown-5m-1775515200,True,0.0000
1,348248,"BNB Up or Down - April 6, 6:40PM-6:45PM ET",bnb-updown-5m-1775515200,True,0.0000
2,316447,Olympique de Marseille vs. FC Metz - Exact Score,fl1-olm-met-2026-04-10-exact-score,True,NaN
3,323929,VfL Bochum vs. Eintracht Braunschweig,bl2-boc-bra-2026-04-12,True,NaN
4,317281,CA Newell's Old Boys vs. CA San Lorenzo de Almagro - More Markets,arg-new-slo-2026-04-12-more-markets,True,NaN
5,344463,"UFC Fight Night: Aljamain Sterling vs. Youssef Zalal (Featherweight, Main Card)",ufc-alj-you-2026-04-25,True,1.0204
6,317279,CA Tucumán vs. CA Tigre - More Markets,arg-cat-tig-2026-04-12-more-markets,True,NaN
7,323335,Genoa CFC vs. US Sassuolo Calcio - Exact Score,sea-gen-sas-2026-04-12-exact-score,True,NaN


### 1.4  Tags / Categories

In [9]:
tags_raw = client.gamma("/tags", limit=20)

print("=== Tags / Categories ===")
display(safe_df(tags_raw, ["id", "label", "slug"]).head(15))

=== Tags / Categories ===


,id,label,slug
0,101259,Health and Human Services,health-and-human-services
1,101842,Sweeden,sweeden
2,101302,attorney general,attorney-general
3,102067,Madrid Open,madrid-open
4,101944,Crypto Summit,crypto-summit
5,100826,DET,det
6,100539,Costello,costello
7,102981,Testing tag,testing-tag
8,101528,altcoin,altcoin
9,102846,Best of 2025,best-of-2025


### 1.5  Public Search

In [10]:
SEARCH_QUERIES = ["presidential", "election", "trump", "fed rate", "bitcoin"]

for query in SEARCH_QUERIES[:2]:          # limit to 2 to stay within rate limits
    results = client.gamma("/public-search", q=query, limit=5)
    if results:
        print(f"\n── Search: '{query}' ({len(results)} results) ──")
        rows = results.get("events", []) if isinstance(results, dict) else results
        display(safe_df(rows, ["id", "question", "slug", "volume24hr"]).head(5))


── Search: 'presidential' (2 results) ──


,id,slug,volume24hr
0,179295,next-president-of-vietnam,416925.446909
1,902772,taiwan-presidential-election-who-will-win,0.000000
2,901930,argentina-presidential-election-who-will-win,0.000000
3,903250,finland-presidential-election-who-will-win,0.000000
4,903276,el-salvador-presidential-election-winner,0.000000



── Search: 'election' (2 results) ──


,id,slug,volume24hr
0,902823,mississippi-gubernatorial-election-presley-d-vs-reeves-r,0.0
1,35723,chile-presidential-election-1st-round-winner,NaN
2,903204,mexican-presidential-election-who-will-win,NaN
3,38539,norwegian-progress-party-frp-ou-22,NaN
4,903527,portugal-legislative-election,NaN


### 1.6  Series & Sports

In [11]:
series_raw = client.gamma("/series", limit=5)
if series_raw:
    print(f"Series count returned: {len(series_raw)}")
    display(safe_df(series_raw, ["id", "title", "slug"]))

sports_raw = client.gamma("/sports", limit=5)
if sports_raw:
    print(f"Sports metadata entries: {len(sports_raw)}")
    display(safe_df(sports_raw, ["id", "title", "slug"]))

Series count returned: 5


,id,title,slug
0,10045,Solana ETF,solana-etf
1,10075,Dogecoin Hit Price Monthly,dogecoin-hit-price-monthly
2,10433,Rocket League,rocket-league
3,5,CPI,cpi
4,10687,Top Magnificent 7,top-magnificent-7


Sports metadata entries: 171


,id
0,1
1,2
2,3
3,95
4,5
...,...
166,158
167,160
168,161
169,128


### 1.7  Lightweight market samples
Use the current `GET /markets` endpoint with a small limit instead of the deprecated sampling endpoints.


In [12]:
sampling_markets = client.gamma(
    "/markets",
    limit=5,
    order="volume24hr",
    ascending="false",
    active="true",
    closed="false",
)
if sampling_markets:
    print("=== Lightweight Market Sample (via /markets) ===")
    display(safe_df(sampling_markets, ["id", "question", "slug"]))

sampling_simplified = client.gamma(
    "/events",
    limit=5,
    active="true",
    closed="false",
    order="volume",
)
if sampling_simplified:
    print("=== Lightweight Event Sample (via /events) ===")
    display(safe_df(sampling_simplified, ["id", "title", "slug"]))


=== Lightweight Market Sample (via /markets) ===


,id,question,slug
0,1640919,US forces enter Iran by April 30?,us-forces-enter-iran-by-april-30-899
1,1394299,US forces enter Iran by December 31?,us-forces-enter-iran-by-december-31-573-642-385-371-179-425-262
2,1455604,Will Trump talk to Xi Jinping in March?,will-trump-talk-to-xi-jinping-in-march-165
3,1856503,Spread: Suns (-10.5),nba-phx-chi-2026-04-05-spread-away-10pt5
4,1706788,US x Iran ceasefire by April 7?,us-x-iran-ceasefire-by-april-7-278


=== Lightweight Event Sample (via /events) ===


,id,title,slug
0,348245,"Dogecoin Up or Down - April 6, 6:40PM-6:45PM ET",doge-updown-5m-1775515200
1,348248,"BNB Up or Down - April 6, 6:40PM-6:45PM ET",bnb-updown-5m-1775515200
2,323929,VfL Bochum vs. Eintracht Braunschweig,bl2-boc-bra-2026-04-12
3,316447,Olympique de Marseille vs. FC Metz - Exact Score,fl1-olm-met-2026-04-10-exact-score
4,317281,CA Newell's Old Boys vs. CA San Lorenzo de Almagro - More Markets,arg-new-slo-2026-04-12-more-markets


## 2.  DATA API
User positions, trades, analytics, leaderboards, open interest.

### 2.1  Trader Leaderboard


In [13]:
leaderboard_raw = client.data(
    "/v1/leaderboard",
    category="OVERALL",   # OVERALL | POLITICS | SPORTS | CRYPTO | CULTURE
    timePeriod="DAY",     # DAY | WEEK | MONTH | ALL
    orderBy="PNL",        # PNL | VOL
    limit=10,
)

print("=== Top 10 Traders (PNL, Today) ===")
display(safe_df(leaderboard_raw, ["rank", "userName", "xUsername", "pnl", "vol"]))

=== Top 10 Traders (PNL, Today) ===


,rank,userName,xUsername,pnl,vol
0,1,CemeterySun,,803396.287791,3.572528e+06
1,2,0x2a2C53bD278c04DA9962Fcf96490E17F3DfB9Bc1-1772479215461,,515457.254631,5.348061e+06
2,3,surfandturf,,466394.673732,1.546153e+06
3,4,VPenguin,Vlad_kori,433930.495592,1.646276e+06
4,5,kch123,,356832.795562,1.628982e+06
5,6,Countryside,,334831.737893,3.297717e+06
6,7,0x8a6C6811e8937F9E8aFc1b9249FA540262c30b3f-1771776258725,,272440.346414,1.488398e+06
7,8,mikesports,,250944.531819,8.294911e+05
8,9,ferrariChampions2026,,240481.266633,3.910576e+06
9,10,SemyonMarmeladov,,232238.863424,1.311619e+06


### 2.2  Open Interest


In [14]:
oi_raw = client.data("/oi")

if isinstance(oi_raw, list) and oi_raw:
    global_oi = oi_raw[0].get("value", "N/A")
    print(f"Global Open Interest: ${float(global_oi):,.2f}" if global_oi != "N/A" else "N/A")
    display(safe_df(oi_raw, ["market", "value", "volume"]).head(10))
elif isinstance(oi_raw, dict):
    print("Open Interest:", oi_raw)


Global Open Interest: $417,702,063.86


,market,value
0,GLOBAL,4.177021e+08


## 3.  CLOB API
Real-time order books, prices, spreads, and trade history.

> **Authentication note:** the endpoints below are all *public read* endpoints.
> Order placement / cancellation requires L2 (wallet-signed) credentials — see https://docs.polymarket.com for the auth flow.

> **Real-time note:** for live updates, subscribe to the WebSocket market channel at `wss://ws-subscriptions-clob.polymarket.com/ws/market` instead of polling these endpoints.



### 3.0  Extract token IDs from the first market


In [16]:
# clobTokenIds arrives as a JSON string from Gamma — parse it first.
# CLOB endpoints always use token_id (NOT condition_id or market id).
token_ids: list[str] = []
if first_market:
    token_ids = parse_token_ids(first_market.get("clobTokenIds", []))
    print(f"Token IDs for '{first_market.get('question', '')[:60]}…':")
    for i, tid in enumerate(token_ids):
        print(f"  [{i}] {tid}")


Token IDs for 'US forces enter Iran by April 30?…':
  [0] 2916184120206223749839849644877707470354946028257066951797428049170871002238
  [1] 76533108781962275310651165149634079251899733930834190485860627580128626747247


### 3.1  Order Book

In [17]:
if token_ids:
    token_id = token_ids[0]          # YES token
    order_book = client.clob(f"/book", token_id=token_id)

    if order_book:
        print(f"\n=== Order Book (token {token_id[:16]}…) ===")
        bids = pd.DataFrame(order_book.get("bids", []))
        asks = pd.DataFrame(order_book.get("asks", []))

        if not bids.empty:
            bids.columns = ["price", "size"]
            bids[["price", "size"]] = bids[["price", "size"]].astype(float)
            print("\nTop 5 Bids:")
            display(bids.sort_values("price", ascending=False).head(5))

        if not asks.empty:
            asks.columns = ["price", "size"]
            asks[["price", "size"]] = asks[["price", "size"]].astype(float)
            print("\nTop 5 Asks:")
            display(asks.sort_values("price").head(5))


=== Order Book (token 2916184120206223…) ===

Top 5 Bids:


,price,size
160,0.997,999533.87
159,0.996,1204277.69
158,0.995,197147.81
157,0.994,16022.60
156,0.993,7984.00



Top 5 Asks:


,price,size
1,0.998,2411495.92
0,0.999,6195804.61


### 3.2  Midpoint & Best Price

In [18]:
if token_ids:
    token_id = token_ids[0]

    midpoint = client.clob("/midpoint", token_id=token_id)
    if midpoint:
        print(f"Midpoint:  {midpoint.get('mid')}")

    buy_price  = client.clob("/price", token_id=token_id, side="BUY")
    sell_price = client.clob("/price", token_id=token_id, side="SELL")
    if buy_price and sell_price:
        print(f"Best BUY:  {buy_price.get('price')}")
        print(f"Best SELL: {sell_price.get('price')}")

Midpoint:  0.9975
Best BUY:  0.997
Best SELL: 0.998


### 3.3  Spread

In [19]:
if token_ids:
    spread = client.clob("/spread", token_id=token_ids[0])
    if spread:
        print(f"Spread: {spread.get('spread')}")


Spread: 0.001


### 3.4  Batch Prices (multiple tokens at once)

In [20]:
if token_ids:
    # Batch prices uses POST /prices with token_id + side pairs.
    price_requests = [
        {"token_id": tid, "side": side}
        for tid in token_ids
        for side in ("BUY", "SELL")
    ]
    prices_raw = client.post("https://clob.polymarket.com/prices", json_data=price_requests)
    if prices_raw:
        print("=== Token Prices ===")
        if isinstance(prices_raw, list):
            display(pd.DataFrame(prices_raw))
        else:
            pprint(prices_raw)


=== Token Prices ===
{'2916184120206223749839849644877707470354946028257066951797428049170871002238': {'BUY': '0.997',
                                                                                  'SELL': '0.998'},
 '76533108781962275310651165149634079251899733930834190485860627580128626747247': {'BUY': '0.002',
                                                                                   'SELL': '0.003'}}


### 3.5  Last Trade Price


In [21]:
if token_ids:
    last_trade = client.clob("/last-trade-price", token_id=token_ids[0])
    if last_trade:
        print(f"Last trade price: {last_trade.get('price')}")

Last trade price: 0.997


### 3.6  Tick Size & Market Info


In [22]:
if first_market:
    condition_id = first_market.get("conditionId") or first_market.get("condition_id")
    if condition_id:
        tick_info = client.clob(f"/tick-size", condition_id=condition_id)
        if tick_info:
            print(f"Tick size: {tick_info.get('minimum_tick_size')}")

        market_info = client.clob(f"/markets/{condition_id}")
        if market_info:
            print("\n── CLOB market info ──")
            pprint({
                k: market_info[k]
                for k in ["condition_id", "tokens", "minimum_order_size",
                          "minimum_tick_size", "is_neg_risk", "accepting_orders"]
                if k in market_info
            })

06:49:10  ERROR     HTTP 400  GET https://clob.polymarket.com/tick-size



── CLOB market info ──
{'accepting_orders': True,
 'condition_id': '0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69dccbde93147ee4443246',
 'minimum_order_size': 5,
 'minimum_tick_size': 0.001,
 'tokens': [{'outcome': 'Yes',
             'price': 0.9975,
             'token_id': '2916184120206223749839849644877707470354946028257066951797428049170871002238',
             'winner': False},
            {'outcome': 'No',
             'price': 0.0025,
             'token_id': '76533108781962275310651165149634079251899733930834190485860627580128626747247',
             'winner': False}]}


### 3.7  Price History

In [23]:
if token_ids:
    # Use the documented time-window filter instead of `resolution`.
    now = int(time.time())
    history = client.clob(
        "/prices-history",
        market=token_ids[0],
        startTs=now - 24 * 60 * 60,
        endTs=now,
        fidelity=100,
    )

    if history and "history" in history:
        hist_df = pd.DataFrame(history["history"])
        if not hist_df.empty:
            hist_df["t"] = pd.to_datetime(hist_df["t"], unit="s", utc=True)
            hist_df["p"] = hist_df["p"].astype(float)
            hist_df = hist_df.rename(columns={"t": "timestamp", "p": "price"})
            print(f"=== Price History ({len(hist_df)} bars, last 24 h) ===")
            display(hist_df.tail(10))


=== Price History (12 bars, last 24 h) ===


,timestamp,price
2,2026-04-05 04:40:50+00:00,0.9970
3,2026-04-05 06:20:01+00:00,0.9975
4,2026-04-05 06:20:53+00:00,0.9975
5,2026-04-05 09:40:45+00:00,0.9985
6,2026-04-05 11:20:53+00:00,0.9975
7,2026-04-05 14:40:49+00:00,0.9975
8,2026-04-05 16:20:51+00:00,0.9975
9,2026-04-05 19:40:50+00:00,0.9975
10,2026-04-05 21:20:49+00:00,0.9975
11,2026-04-05 22:48:45+00:00,0.9975


### 3.8  Fee Rate & Neg-Risk Flag

In [24]:
if token_ids:
    token_id = token_ids[0]

    neg_risk_info = client.clob("/neg-risk", token_id=token_id)
    if neg_risk_info:
        print(f"Neg-risk info for token {token_id[:16]}…:", neg_risk_info)

    fee_rate = client.clob("/fee-rate", token_id=token_id)
    if fee_rate:
        print(f"Maker fee rate: {fee_rate.get('maker_fee_rate')}  "
              f"Taker fee rate: {fee_rate.get('taker_fee_rate')}")
else:
    print("Skipping neg-risk and fee-rate lookups because no token IDs were found.")

Neg-risk info for token 2916184120206223…: {'neg_risk': False}
Maker fee rate: None  Taker fee rate: None
